## A MoE Transformer with a Fused Pallas Kernel, written in JAX

Hyperparams

* `n_layers` = 12
*  `d_model` = 256
*  `d_ff` = 512
*  `n_heads` = 16
*  `n_kv_heads` = 2
*  `d_qkv` = 16
*  `n_embed` = 13
*  `n_experts` = 8
*  `n_active` = 2

Total params: $2D(N+K)HL + 2DEFL + 2DV + DEL = 2.70e7$ (~27M)

Active params: $2D(N+K)HL + 2DkFL + 2DV + DEL = 8.09e6$ (~8M), where $k$ denotes number of active experts

In [1]:
!pip install --upgrade jax libtpu
# !pip install --upgrade jax


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import jax
import jax.numpy as jnp
from jax import lax, grad, random
from jax.sharding import PartitionSpec as P
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

from flax import struct
import optax

import numpy as np
from functools import partial
import timeit

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
# jax.config.update("jax_num_cpu_devices", 8) # simulate 8 devices -> disable later

jax.devices()

E0000 00:00:1787898207.698488     176 common_lib.cc:947] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:239


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0),
 TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0),
 TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0),
 TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0),
 TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

In [4]:
X, Y = 8, 1

Explicit = jax.sharding.AxisType.Explicit

mesh = jax.make_mesh((X, Y), ('X', 'Y'), (Explicit, Explicit)) # default Explicit
jax.set_mesh(mesh)

## Building the model

In [5]:
@struct.dataclass
class Config:
    n_layers: jnp.int32

    d_model: jnp.int32
    d_ff: jnp.int32

    n_heads: jnp.int32
    n_kv_heads: jnp.int32
    d_qkv: jnp.int32

    n_experts: jnp.int32
    n_active: jnp.int32
    
    n_embed: jnp.int32
    max_seq_len: jnp.int32
    
    dtype: jnp.dtype = jnp.bfloat16
    param_dtype: jnp.dtype = jnp.float32

In [6]:
def init_params(cfg: Config, rng: random.key) -> dict:
    
    embed_key, unembed_key, rng = random.split(rng, 3)

    params = {}
    
    params["W_embed"] = random.normal(embed_key, (cfg.n_embed, cfg.d_model), dtype=cfg.param_dtype) * 0.02
    params["W_unembed"] = random.normal(unembed_key, (cfg.d_model, cfg.n_embed), dtype=cfg.param_dtype) * 0.02

    initializer = jax.nn.initializers.glorot_normal()
    initializer_attn = jax.nn.initializers.glorot_normal(in_axis=0, out_axis=(1, 2))
    initializer_proj = jax.nn.initializers.glorot_normal(in_axis=(0, 1), out_axis=2)
    initializer_ffw = jax.nn.initializers.glorot_normal(in_axis=1, out_axis=2)
    
    params["layers"] = []
    for i in range(cfg.n_layers):
        layer_params = {}
        
        q_key, k_key, v_key, o_key, route_key, in_key, out_key, rng = random.split(rng, 8)
        
        layer_params["W_q"] = initializer_attn(q_key, (cfg.d_model, cfg.n_heads, cfg.d_qkv), dtype=cfg.param_dtype)
        layer_params["W_k"] = initializer_attn(k_key, (cfg.d_model, cfg.n_kv_heads, cfg.d_qkv), dtype=cfg.param_dtype)
        layer_params["W_v"] = initializer_attn(v_key, (cfg.d_model, cfg.n_kv_heads, cfg.d_qkv), dtype=cfg.param_dtype)
        
        layer_params["W_o"] = initializer_proj(o_key, (cfg.n_heads, cfg.d_qkv, cfg.d_model), dtype=cfg.param_dtype)

        layer_params["W_r"] = initializer(route_key, (cfg.d_model, cfg.n_experts), dtype=cfg.param_dtype)
        
        layer_params["W_in"] = initializer_ffw(in_key, (cfg.n_experts, cfg.d_model, cfg.d_ff), dtype=cfg.param_dtype)
        layer_params["W_out"] = initializer_ffw(out_key, (cfg.n_experts, cfg.d_ff, cfg.d_model), dtype=cfg.param_dtype)

        layer_params["gamma1"] = jnp.ones((cfg.d_model, ), dtype=cfg.param_dtype)
        layer_params["beta1"] = jnp.zeros((cfg.d_model, ), dtype=cfg.param_dtype)

        layer_params["gamma2"] = jnp.ones((cfg.d_model, ), dtype=cfg.param_dtype)
        layer_params["beta2"] = jnp.zeros((cfg.d_model, ), dtype=cfg.param_dtype)

        params["layers"].append(layer_params)

    # final LN

    params["gamma"] = jnp.ones((cfg.d_model, ), dtype=cfg.param_dtype)
    params["beta"] = jnp.zeros((cfg.d_model, ), dtype=cfg.param_dtype)
    
    return params

In [7]:
def RoPE(x: jnp.array) -> jnp.array:

    H = x.shape[-1]
    T = x.shape[1]
    
    assert H % 2 == 0

    base = 10000
    exp = -2 * jnp.arange(H//2, dtype=x.dtype) / H
    theta = jnp.arange(T).reshape(T, 1) * jnp.float_power(base, exp)

    cos_vec = jnp.cos(theta).reshape(1, T, 1, H//2)
    sin_vec = jnp.sin(theta).reshape(1, T, 1, H//2)

    x1, x2 = jnp.split(x, 2, axis=-1)

    x1_new = x1 * cos_vec - x2 * sin_vec
    x2_new = x1 * sin_vec + x2 * cos_vec

    return jnp.concatenate([x1_new, x2_new], axis=-1, dtype=x.dtype)

In [8]:
def attention(x: jnp.array, W_q: jnp.array, W_k: jnp.array, W_v: jnp.array, W_o: jnp.array, attn_mask: jnp.array = None, is_causal: bool = True) -> jnp.array:

    B = x.shape[0]
    T = x.shape[1] # assume T == S
    
    N = W_q.shape[1]
    K = W_k.shape[1]
    H = W_q.shape[-1]

    assert N % K == 0

    W_q = W_q.astype(x.dtype)
    W_k = W_k.astype(x.dtype)
    W_v = W_v.astype(x.dtype)
    W_o = W_o.astype(x.dtype)
    
    
    q = jnp.einsum('btd,dnh->btnh', x, W_q)
    k = jnp.einsum('bsd,dnh->bsnh', x, W_k)
    v = jnp.einsum('bsd,dnh->bsnh', x, W_v)

    q = RoPE(q)
    k = RoPE(k)

    q = q.reshape((B, T, K, -1, H))

    s = jnp.einsum('btkgh,bskh->btskg', q, k)

    s /= jnp.sqrt(H)
    
    if (attn_mask is not None or is_causal): 
    
        mask = jnp.ones((B, T, T), dtype=x.dtype)

        if (is_causal):
            mask = jnp.tril(mask)
        
        if (attn_mask is not None):
            mask = mask * attn_mask.reshape((B, -1, T))

        mask = mask.reshape((B, T, T, 1, 1))
        
        s = jnp.where(mask, s, -jnp.inf)

    # s = jnp.exp(s) / jnp.sum(jnp.exp(s), axis=2, keepdims=True) # softmax
    s = jax.nn.softmax(s, axis=2)

    a = jnp.einsum('btskg,bskh->btkgh', s, v)
    a = a.reshape((B, T, N, H))

    out = jnp.einsum('btnh,nhd->btd', a, W_o)
    return out

In [9]:
# megablocks GMM approach


def MoE_proj(x, W_in, W_out, idx, blk_F, block_size=256):
    B, D = x.shape
    E, _, F = W_in.shape
    
    # block_size = 256 # multiple of 8
    n_blocks = B // block_size
    n_tiles = n_blocks + E - 1

    # preprocessing
    
    experts = jnp.zeros((n_tiles,), dtype=jnp.int32)
    indices = jnp.zeros((n_tiles,), dtype=jnp.int32)
    masks = jnp.ones((n_tiles, block_size), dtype=jnp.int32)

    idx = jnp.pad(idx, (0, block_size))
    x = jnp.pad(x, ((0, block_size), (0, 0)))
    
    def fun(i, carrys):
        j, cur_expert, experts, indices, masks = carrys

        cur_idxs = lax.dynamic_slice(idx, (j,), (block_size,))
        
        indices = indices.at[i].set(j)
        experts = experts.at[i].set(jnp.minimum(cur_expert, E-1))
        masks = lax.dynamic_update_slice(masks, 
                                        jnp.where(cur_idxs == cur_expert, 1, 0).reshape(-1, block_size),
                                        (i, 0))

        j, new_expert = lax.cond(
            idx[j+block_size-1] == cur_expert,
            lambda j,e: (j+block_size, e),
            lambda j,e: (j, e+1),
            j, cur_expert
        )
        new_expert = lax.cond(
            idx[j] > new_expert,
            lambda e: idx[j],
            lambda e: e,
            new_expert
        )

        return j, new_expert, experts, indices, masks

    _, _, experts, indices, masks = lax.fori_loop( # indices = element indices
        0,
        n_tiles,
        fun,
        (0, 0, experts, indices, masks)
    )


    # grid -> 2 loops
    # outer loop over tiles, inner loop over F
    

    def MoE_kernel(
        experts, indices, # prefetched scalars
        x_ref, W_in_ref, W_out_ref, mask_ref, # input refs,
        o_ref, # output ref
        accum_scratch, # scratch buffer
    ):
        del experts

        is_start = pl.program_id(1) == 0
        first_block = pl.program_id(0) == 0
        new_block = indices[jnp.maximum(0, pl.program_id(0)-1)] != indices[pl.program_id(0)]
        
        @pl.when(is_start & (first_block | new_block))
        def _():
            o_ref[...] = jnp.zeros_like(o_ref)

        accum_scratch[...] = lax.dot(
            lhs=jax.nn.gelu(
                lax.dot(
                    lhs=x_ref[...],
                    rhs=W_in_ref[0,:,:],
                    preferred_element_type=jnp.float32,
                )
            ).astype(x_ref.dtype),
            rhs=W_out_ref[0,:,:],
            preferred_element_type=jnp.float32,
        )

        o_ref[...] += (accum_scratch[...] * mask_ref[0, :, :]).astype(o_ref.dtype)

    
    def x_map(i, j, experts, indices):
        del j, experts
        return (indices[i] // block_size, 0)

    def W_in_map(i, j, experts, indices):
        del indices
        return (experts[i], 0, j)

    def W_out_map(i, j, experts, indices):
        del indices
        return (experts[i], j, 0)

    def mask_map(i, j, experts, indices):
        del j, experts, indices
        return (i, 0, 0)

    def o_map(i, j, experts, indices):
        del j, experts
        return (indices[i] // block_size, 0)

    masks = lax.broadcast_in_dim(masks, (*masks.shape, D), broadcast_dimensions=(0,1))
    
    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=2,
        grid=(n_tiles, F // blk_F),
        in_specs=[
            pl.BlockSpec((block_size, D), index_map=x_map),
            pl.BlockSpec((1, D, blk_F), index_map=W_in_map),
            pl.BlockSpec((1, blk_F, D), index_map=W_out_map),
            pl.BlockSpec((1, block_size, D), index_map=mask_map)
        ],
        out_specs=pl.BlockSpec((block_size, D), index_map=o_map),
        scratch_shapes=[
            pltpu.VMEM((block_size, D), dtype=jnp.float32)
        ]
    )

    kernel = pl.pallas_call(
        MoE_kernel,
        grid_spec=grid_spec,
        out_shape=jax.ShapeDtypeStruct.like(x)
    )

    out = kernel(experts, indices, x, W_in, W_out, masks)

    out = out.at[:-block_size].get()
    
    return out


@jax.jit(
    static_argnames=["k", "blk_F", "block_size"]
)
def mlp_with_kernel(x: jnp.array, W_r: jnp.array, k: jnp.int32, W_in: jnp.array, W_out: jnp.array, blk_F: jnp.int32, block_size: jnp.int32) -> jnp.array:
    
    B = x.shape[0]
    T = x.shape[1]
    D = x.shape[2]
    E = W_in.shape[0]
    F = W_in.shape[2]
    
    @jax.shard_map(
        in_specs=(P('X'), P('X', None), P(None, None, None), P(None, None, None)),
        out_specs=P('X', None),
        check_vma=False
    )
    def local_ragged_dot(idx, x, W_in, W_out): 
        perm = jnp.argsort(idx)
        x_sorted = x[perm]
        idx_sorted = idx[perm]

        out_sorted = MoE_proj(x_sorted, W_in, W_out, idx_sorted, blk_F, block_size)
        
        perm_inv = jnp.argsort(perm)
        
        out = out_sorted[perm_inv]
        
        return out
    
    W_r = W_r.astype(x.dtype) # consider if needed later
    W_in = W_in.astype(x.dtype)
    W_out = W_out.astype(x.dtype)
    
    routing = jnp.einsum('btd,de->bte', x, W_r)
    routing = jax.nn.softmax(routing) # in case we end up with 0
    
    gate, idx = lax.top_k(routing, k)

    gate = gate / gate.sum(axis=-1, keepdims=True, dtype=jnp.float32) # temporarily set as float32 for greater precision
    gate = gate.astype(x.dtype)
    
    idx_flat = idx.flatten()
    
    x_repeat = jnp.repeat(x, k, axis=1)
    x_repeat = x_repeat.reshape((-1, D))

    out_repeat = local_ragged_dot(idx_flat, x_repeat, W_in, W_out)
    out_repeat = out_repeat.reshape((B, T, -1, D))

    out = jnp.einsum('btkd,btk->btd', out_repeat, gate)

    return out

In [10]:
@jax.custom_vjp
def ragged_dot(x, W, g):
    return lax.ragged_dot(x, W, g)

def ragged_dot_fwd(x, W, g):
    return ragged_dot(x, W, g), (x, W, g)

def ragged_dot_bwd(carrys, dOut):
    x, W, g = carrys

    dx = lax.ragged_dot(dOut, jnp.transpose(W, (0,2,1)), g)
    
    dW = lax.ragged_dot_general(
        x, dOut, g,
        lax.RaggedDotDimensionNumbers(
            (([0], [0]), ([], [])),
            [0],
            []
        )
    )

    dW = lax.psum(dW, 'X')
    
    return dx, dW, None

ragged_dot.defvjp(ragged_dot_fwd, ragged_dot_bwd)

In [11]:
@jax.jit(
    static_argnames=["k"]
)
def mlp(x: jnp.array, W_r: jnp.array, k: jnp.int32, W_in: jnp.array, W_out: jnp.array) -> jnp.array:
    
    B = x.shape[0]
    T = x.shape[1]
    D = x.shape[2]
    E = W_in.shape[0]

    
    @jax.shard_map(
        in_specs=(P('X'), P('X', None), P(None, None, None), P(None, None, None)),
        out_specs=P('X', None)
    )
    def local_ragged_dot(idx, x, W_in, W_out): 
        perm = jnp.argsort(idx)
        x_sorted = x[perm]
    
        group_sizes = jnp.bincount(idx, length=E)
        tmp = ragged_dot(x_sorted, W_in, group_sizes)
        act = jax.nn.gelu(tmp)
        
        out_sorted = ragged_dot(act, W_out, group_sizes)
        perm_inv = jnp.argsort(perm)    
        out = out_sorted[perm_inv]
        
        return out

    
    W_r = W_r.astype(x.dtype) # consider if needed later
    W_in = W_in.astype(x.dtype)
    W_out = W_out.astype(x.dtype)
    
    routing = jnp.einsum('btd,de->bte', x, W_r)
    routing = jax.nn.softmax(routing) # in case we end up with 0
    
    gate, idx = lax.top_k(routing, k)

    gate = gate / gate.sum(axis=-1, keepdims=True, dtype=jnp.float32) # temporarily set as float32 for greater precision
    gate = gate.astype(x.dtype)
    
    idx_flat = idx.flatten()
    
    x_repeat = jnp.repeat(x, k, axis=1)
    x_repeat = x_repeat.reshape((-1, D))

    out_repeat = local_ragged_dot(idx_flat, x_repeat, W_in, W_out)
    out_repeat = out_repeat.reshape((B, T, -1, D))

    out = jnp.einsum('btkd,btk->btd', out_repeat, gate)

    return out

In [12]:
cfg = Config(
    n_layers = 1,
    # d_model = 256,
    # d_ff = 512,
    # d_model = 1024,
    # d_ff = 4096,

    d_model = 2048,
    d_ff = 8192,
    
    n_heads = 16,
    n_kv_heads = 2,
    d_qkv = 16,
    n_experts = 8,
    n_active = 2,
    n_embed = 13,
    max_seq_len = 16,
)

params = init_params(cfg, random.key(42))

In [13]:
B = 2048
T = cfg.max_seq_len
D = cfg.d_model
F = cfg.d_ff
E = cfg.n_experts
k = 2

# x = jnp.arange((B * T)).reshape((B, T))
# x = x % cfg.n_embed

x = random.uniform(random.key(42), (B, T, D), dtype=cfg.dtype)
x = jax.device_put(x, P('X', None, None))

# x = jnp.arange(T).repeat(D).reshape(T,D)

# x = jnp.arange(T).repeat(D).reshape((1,T,D)).repeat(B, axis=0)


layer_params = params["layers"][0]

W_r = layer_params["W_r"]
W_in = layer_params["W_in"]
W_out = layer_params["W_out"]

In [14]:
def benchmark(f, n_trials=100):
    def run(*args, **kwargs):
    
        # compile func first

        jax.block_until_ready(f(*args, **kwargs))

        res = timeit.timeit(lambda: jax.block_until_ready(f(*args, **kwargs)), number=n_trials)
        time = res / n_trials

        return time
    return run


out = mlp_with_kernel(x, W_r, k, W_in, W_out, blk_F=F//16, block_size=256)
ref = mlp(x, W_r, k, W_in, W_out)

np.testing.assert_allclose(
    out,
    ref,
    atol=1e-2,
    rtol=1e-2,
)

# Comparing MLP time
print(f"MLP with lax.ragged_dot: {benchmark(mlp)(x, W_r, k, W_in, W_out):.3e}")
print(f"MLP with custom kernel: {benchmark(mlp_with_kernel)(x, W_r, k, W_in, W_out, blk_F=F//16, block_size=256):.3e}")

MLP with lax.ragged_dot: 9.797e-03
MLP with custom kernel: 8.139e-03


In [31]:
# Sweeping F/D for fixed D:

D = 2048

multiples = [0.125, 0.25, 0.5, 1, 2, 4, 8]

for multiple in multiples:
    
    F = int(D * multiple)

    print(f"SWEEPING D={D}, F={F}...")
    
    cfg = Config(
        n_layers = 1,
        d_model = D,
        d_ff = F,
        n_heads = 16,
        n_kv_heads = 2,
        d_qkv = 16,
        n_experts = 8,
        n_active = 2,
        n_embed = 13,
        max_seq_len = 16,
    )

    ragged_dot_times = []
    kernel_times = []

    
    num_blks_F = int(np.pow(2, np.floor(np.log2(2*D*F / 2e6))))
    blk_F = F // max(1, num_blks_F)
    
    block_size = 256
    
    print(f"USING blk_F = {blk_F}, block_size = {block_size}")
    
    for key in random.split(random.key(42), 5):

        params_key, x_key = random.split(key)
        
        params = init_params(cfg, params_key)

        B = 2048
        T = cfg.max_seq_len
        D = cfg.d_model
        E = cfg.n_experts
        k = 2
        
        x = random.uniform(x_key, (B, T, D), dtype=cfg.dtype)
        x = jax.device_put(x, P('X', None, None))

        layer_params = params["layers"][0]
        
        W_r = layer_params["W_r"]
        W_in = layer_params["W_in"]
        W_out = layer_params["W_out"]

        out = mlp_with_kernel(x, W_r, k, W_in, W_out, blk_F=blk_F, block_size=block_size)
        ref = mlp(x, W_r, k, W_in, W_out)
        
        np.testing.assert_allclose(
            out,
            ref,
            atol=1e-2,
            rtol=1e-2,
        )

        ref_time = benchmark(mlp)(x, W_r, k, W_in, W_out)
        out_time = benchmark(mlp_with_kernel)(x, W_r, k, W_in, W_out, blk_F=blk_F, block_size=block_size)

        ragged_dot_times.append(ref_time)
        kernel_times.append(out_time)

    ref_time, ref_std = np.mean(ragged_dot_times), np.std(ragged_dot_times)
    out_time, out_std = np.mean(kernel_times), np.std(kernel_times)
        
    # Comparing MLP time
    print(f"MLP with lax.ragged_dot: {ref_time:.3e} ± {ref_std:.3e}")
    print(f"MLP with custom kernel: {out_time:.3e} ± {out_std:.3e}")    
    print("-------------------------------------")
    print()

SWEEPING D=2048, F=256...
USING blk_F = 256, block_size = 256
MLP with lax.ragged_dot: 1.368e-03 ± 3.094e-05
MLP with custom kernel: 1.515e-03 ± 8.893e-06
-------------------------------------

SWEEPING D=2048, F=512...
USING blk_F = 512, block_size = 256
MLP with lax.ragged_dot: 1.507e-03 ± 4.678e-05
MLP with custom kernel: 2.025e-03 ± 1.329e-05
-------------------------------------

SWEEPING D=2048, F=1024...
USING blk_F = 512, block_size = 256
MLP with lax.ragged_dot: 2.093e-03 ± 8.095e-05
MLP with custom kernel: 2.419e-03 ± 2.261e-05
-------------------------------------

SWEEPING D=2048, F=2048...
USING blk_F = 512, block_size = 256
MLP with lax.ragged_dot: 2.823e-03 ± 1.726e-05
MLP with custom kernel: 3.318e-03 ± 7.039e-05
-------------------------------------

SWEEPING D=2048, F=4096...
USING blk_F = 512, block_size = 256
MLP with lax.ragged_dot: 4.944e-03 ± 5.679e-05
MLP with custom kernel: 4.909e-03 ± 1.018e-04
-------------------------------------

SWEEPING D=2048, F=8192...


In [ ]:
def layerNorm(x: jnp.array, gamma: jnp.array, beta: jnp.array) -> jnp.array:

    mu = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)

    epsilon = 1e-5
    tmp = (x - mu) / jnp.sqrt(var + epsilon)

    out = gamma * tmp + beta
    
    return out.astype(x.dtype)

In [ ]:
def transformer_block(layer_params: optax.Params, x: jnp.array, k: jnp.int32, attn_mask: jnp.array = None) -> jnp.array:

    # pre-norm
    
    gamma1 = layer_params["gamma1"]
    beta1 = layer_params["beta1"]
    
    x_norm = layerNorm(x, gamma1, beta1)
    
    # attention
    
    W_q = layer_params["W_q"]
    W_k = layer_params["W_k"]
    W_v = layer_params["W_v"]
    W_o = layer_params["W_o"]
    
    attn_out = attention(x_norm, W_q, W_k, W_v, W_o, attn_mask=attn_mask, is_causal=True)

    attn_out = attn_out + x # residue connection

    # norm

    gamma2 = layer_params["gamma2"]
    beta2 = layer_params["beta2"]
    
    attn_norm = layerNorm(attn_out, gamma2, beta2)

    # mlp

    W_r = layer_params["W_r"]
    W_in = layer_params["W_in"]
    W_out = layer_params["W_out"]

    ffw_out = mlp(attn_norm, W_r, k, W_in, W_out)

    ffw_out = ffw_out + attn_out # residue connection
    
    return ffw_out

In [ ]:
def forward(params: optax.Params, x: jnp.array, k: jnp.int32, attn_mask: jnp.array = None) -> jnp.array:

    W_embed = params["W_embed"].astype(x.dtype)
    
    x = jnp.einsum("btv,vd->btd", x, W_embed) # embedding

    
    for layer_params in params["layers"]:
        x = transformer_block(layer_params, x, k, attn_mask=attn_mask)

    gamma = params["gamma"]
    beta = params["beta"]
    
    x_norm = layerNorm(x, gamma, beta)
    
    W_unembed = params["W_unembed"].astype(x.dtype)
    
    out = jnp.einsum("btd,dv->btv", x_norm, W_unembed) # unembedding

    return out